## 0. Install dependencies

Run this cell once. `%pip` installs into whatever Python is running this notebook.

In [8]:
%pip install pandas beautifulsoup4 lxml

Note: you may need to restart the kernel to use updated packages.


# Clean and transform `raw_data.csv`

Targeted cleaning of the BookXcess scrape produced by `main_crawler.ipynb`.

**Steps**
1. Load the CSV and audit for nulls
2. Rename columns and prep datatypes
3. Strip HTML tags out of the product description
4. Remove `[Bargain Corner]` prefix from titles
5. Clean recurring `?` characters and stray whitespace across all text fields
6. Validate `author` and `publisher` are real names — drop rows that are numbers or symbols; strip dangling punctuation
7. Calculate discount; drop rows where the discount is illogical (negative, undefined)
8. Final checks and save to `cleaned_data_2.csv`

In [9]:
import re
import html
import unicodedata
import pandas as pd
from bs4 import BeautifulSoup

INPUT_FILE  = "raw_data.csv"
OUTPUT_FILE = "cleaned_data.csv"

# Currency is the same on every row — keep as a constant, not a column.
CURRENCY = "MYR"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 1. Load the raw CSV and audit for nulls

In [10]:
df = pd.read_csv(INPUT_FILE)

print("Shape:", df.shape)
print()
print("Data types:")
print(df.dtypes)
print()
print("Missing values per column:")
print(df.isna().sum())
print()
df.head(3)

Shape: (154714, 13)

Data types:
Product ID            int64
Title                   str
Product Type            str
Vendor                  str
SKU                     str
Current Price       float64
Compare at Price    float64
Currency                str
Grams                 int64
Tags                    str
Body HTML               str
Image URL               str
Created At              str
dtype: object

Missing values per column:
Product ID             0
Title                  1
Product Type        3121
Vendor               169
SKU                    4
Current Price          0
Compare at Price     958
Currency               0
Grams                  0
Tags                   3
Body HTML           2632
Image URL              7
Created At             0
dtype: int64



,Product ID,Title,Product Type,Vendor,SKU,Current Price,Compare at Price,Currency,Grams,Tags,Body HTML,Image URL,Created At
0,6978035122382,Visions,Kelley Armstrong,Sphere,9780751547238,17.9,62.90,MYR,355,"[2022], BXOS, F CTM, Fiction, Kelley Armstrong...","<p>Olivia Jones is smart, capable, loyal...and...",https://cdn.shopify.com/s/files/1/0601/0883/29...,2022-04-01T10:46:08+08:00
1,6978035318990,The Bone Bed,Patricia Cornwell,Sphere,9780751548174,17.9,43.95,MYR,360,"[2022], BXOS, F CTM, Fiction, Paperback, Patri...",<p>A woman has vanished while digging a dinosa...,https://cdn.shopify.com/s/files/1/0601/0883/29...,2022-04-01T10:46:08+08:00
2,6978035417294,Jasmine Skies,Sita Brahmachari,MacMillan Children's Books,9781447205180,12.9,32.95,MYR,340,"[2022], BXOS, F YA, Fiction, Paperback, RM 10 ...",<p>Mira Levenson is bursting with excitement a...,https://cdn.shopify.com/s/files/1/0601/0883/29...,2022-04-01T10:46:09+08:00


In [11]:

print("df.shape before dropping NaNs:", df.shape)
df.dropna(inplace=True)
print("df.shape after dropping NaNs:", df.shape)

df.shape before dropping NaNs: (154714, 13)
df.shape after dropping NaNs: (148113, 13)


In [12]:
df.isna().sum()

Product ID          0
Title               0
Product Type        0
Vendor              0
SKU                 0
Current Price       0
Compare at Price    0
Currency            0
Grams               0
Tags                0
Body HTML           0
Image URL           0
Created At          0
dtype: int64

## 2. Rename columns and prep datatypes

- `Product Type` actually contains the **author name** → rename to `author`.
- `Vendor` is the **publisher** → rename to `publisher`.
- Lowercase everything and use underscores instead of spaces.
- Drop the constant `Currency` column.
- Parse `Created At` into a real datetime.

In [13]:
df = df.rename(columns={
    "Product ID":       "product_id",
    "Title":            "title",
    "Product Type":     "author",
    "Vendor":           "publisher",
    "SKU":              "sku",
    "Current Price":    "current_price",
    "Compare at Price": "compare_price",
    "Currency":         "currency",
    "Grams":            "grams",
    "Tags":             "tags",
    "Body HTML":        "body_html",
    "Image URL":        "image_url",
    "Created At":       "created_at",
})

# Sanity-check before dropping.
assert df["currency"].dropna().nunique() == 1, "Expected only MYR"
df = df.drop(columns=["currency"])

df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
print("created_at dtype:", df["created_at"].dtype)
print("Date range:", df["created_at"].min(), "to", df["created_at"].max())
print("Columns:", df.columns.tolist())

created_at dtype: datetime64[us, UTC+08:00]
Date range: 2022-04-01 10:46:04+08:00 to 2026-05-05 14:27:17+08:00
Columns: ['product_id', 'title', 'author', 'publisher', 'sku', 'current_price', 'compare_price', 'grams', 'tags', 'body_html', 'image_url', 'created_at']


## 3. Strip HTML tags from the product description

`body_html` looks like `<p>...text...</p>`. BeautifulSoup turns it into plain text reliably
without fragile regex. Takes about 1–2 minutes on ~150k rows.

In [14]:
def html_to_text(value):
    """Strip HTML tags, return plain text. Returns None for missing values."""
    if pd.isna(value):
        return None
    soup = BeautifulSoup(value, "lxml")
    text = soup.get_text(separator=" ")
    text = re.sub(r"\s+", " ", text)
    return text.strip() or None


df["description"] = df["body_html"].apply(html_to_text)
df = df.drop(columns=["body_html"])

print("Sample descriptions after stripping HTML:")
for text in df["description"].dropna().head(3):
    print("-", text[:120], "...")
print()
print("Missing descriptions:", df["description"].isna().sum())

Sample descriptions after stripping HTML:
- Olivia Jones is smart, capable, loyal...and the daughter of convicted serial killers. Taking refuge in the mysterious to ...
- A woman has vanished while digging a dinosaur bone bed in the remote wilderness of Canada. Somehow, the only evidence ha ...
- Mira Levenson is bursting with excitement as she flies to India to stay with her aunt and cousin for the first time. As  ...

Missing descriptions: 4


## 4. Remove `[Bargain Corner]` prefix from titles

Some titles in the raw data are prefixed with `[Bargain Corner]` (or lowercase variants like
`[Bargain corner]`). These are store-category labels, not part of the actual book title.
We strip them with a case-insensitive regex anchored to the start of the string.

In [15]:
_BARGAIN_RE = re.compile(r"^\[bargain\s+corner\]\s*", re.IGNORECASE)

before_count = df["title"].str.contains(_BARGAIN_RE, na=False).sum()
print(f"Titles with '[Bargain Corner]' prefix before cleaning: {before_count}")

df["title"] = df["title"].apply(
    lambda v: _BARGAIN_RE.sub("", v).strip() if isinstance(v, str) else v
)

after_count = df["title"].str.contains(_BARGAIN_RE, na=False).sum()
print(f"Titles with '[Bargain Corner]' prefix after cleaning:  {after_count}")

Titles with '[Bargain Corner]' prefix before cleaning: 2323
Titles with '[Bargain Corner]' prefix after cleaning:  0


## 5. Clean recurring `?` characters and stray whitespace across all text fields

Applied to every text column:
1. Decode HTML entities (`&amp;` → `&`, `&#10;` → newline, etc.).
2. NFKC unicode normalisation — converts full-width chars to ASCII, decomposes ligatures.
3. Replace newlines / tabs / non-breaking-spaces with a regular space.
4. Remove sequences of **two or more consecutive `?`** (e.g. `?????` is a data artefact).
5. Collapse multiple spaces into one and strip leading / trailing whitespace.
6. Replace any resulting empty strings with `None`.

In [16]:
def clean_text(value):
    """Decode entities, normalize unicode, strip recurring ? and tidy whitespace."""
    if pd.isna(value):
        return value
    text = str(value)
    # Normalize unicode (full-width → half-width, ligatures, etc.)
    text = unicodedata.normalize("NFKC", text)
    # Decode &amp;, &quot;, &#10;, etc.
    text = html.unescape(text)
    # Replace newline / carriage-return / tab / non-breaking-space with a regular space
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    text = text.replace("\u00a0", " ")
    # Remove runs of 2+ consecutive question marks (data artefact)
    text = re.sub(r"\?{2,}", "", text)
    # Collapse whitespace and strip ends
    text = re.sub(r"\s+", " ", text).strip()
    return text or None


TEXT_COLUMNS = ["title", "author", "publisher", "sku", "tags", "image_url", "description"]

for col in TEXT_COLUMNS:
    df[col] = df[col].apply(clean_text)

# Treat empty strings as missing (belt-and-suspenders after clean_text already returns None)
df[TEXT_COLUMNS] = df[TEXT_COLUMNS].replace({"": None})

# Quick check: no tabs or newlines should survive
for col in TEXT_COLUMNS:
    bad = df[col].str.contains(r"[\n\r\t]", regex=True, na=False).sum()
    if bad:
        print(f"WARNING: {col} still has {bad} rows with newlines/tabs")

print("Text cleaning done. Sample titles:")
for t in df["title"].dropna().head(5):
    print(" -", t)

Text cleaning done. Sample titles:
 - Visions
 - The Bone Bed
 - Jasmine Skies
 - The Old Devils
 - PHILIP K. DICK'S ELECTRIC DREAMS: VOLUME 1


## 6. Validate `author` and `publisher` are real names

A valid name must contain **at least one letter** (a–z). Rows where `author` or `publisher`
is a pure number, a string of symbols, or only punctuation are not usable and are dropped.

Before checking, we strip **dangling punctuation** from both ends (e.g. a leading `!` or
trailing `,` that crept in during scraping).

In [17]:
_DANGLING_RE = re.compile(r"^[\s!,;:.\-&]+|[\s!,;:.\-&]+$")


def strip_dangling_punctuation(value):
    """Remove leading/trailing stray punctuation characters."""
    if pd.isna(value):
        return value
    cleaned = _DANGLING_RE.sub("", value).strip()
    return cleaned if cleaned else None


def is_valid_name(value):
    """Return True if value contains at least one letter. NaN is treated as valid (missing ≠ bad)."""
    if pd.isna(value):
        return True
    return bool(re.search(r"[a-zA-Z]", value))


# Strip dangling punctuation first, then check validity.
df["author"]    = df["author"].apply(strip_dangling_punctuation)
df["publisher"] = df["publisher"].apply(strip_dangling_punctuation)

invalid_author    = ~df["author"].apply(is_valid_name)
invalid_publisher = ~df["publisher"].apply(is_valid_name)
invalid_either    = invalid_author | invalid_publisher

print("Rows with invalid author:   ", int(invalid_author.sum()))
print("Rows with invalid publisher:", int(invalid_publisher.sum()))
print("Rows dropped (either):      ", int(invalid_either.sum()))

if invalid_either.sum() > 0:
    print("\nSamples of dropped rows:")
    print(df.loc[invalid_either, ["title", "author", "publisher"]].head(10).to_string())

before = len(df)
df = df[~invalid_either].reset_index(drop=True)
print(f"\nDropped {before - len(df)} rows. Remaining: {len(df)}")

Rows with invalid author:    126
Rows with invalid publisher: 119
Rows dropped (either):       130

Samples of dropped rows:
                                                                   title author           publisher
25036  Gift Cards: Crafted With Care All Occasion Card Collection (10Pc)      0   Crafted With Care
26744                                        Notebook: Bunyip A5 Journal      0  Hardie Grant Books
27523                     Notebook: Make Do And Mend (3X Mini Notebooks)      0          Kyle Books
27705                                        Tea Time : 3Xmini Notebooks      0          Kyle Books
60167                                                 小小军事迷 大型手工仿真书 陆战霸主   格林图书             长江少年出版社
60175                                                 小小军事迷 大型手工仿真书 -护卫舰   格林图书           长江少年儿童出版社
60188                                                           本草纲目(全套)    李时珍            黑龙江美术出版社
60205                                                      多面金剛‧驅逐艦(簡體書)   

## 7. Calculate discount and drop illogical price rows

- `discount_pct = (compare_price - current_price) / compare_price × 100`
- **Drop** rows where `compare_price < current_price` — negative discount means the data is wrong.
- **Drop** rows where `compare_price == 0` — division by zero gives undefined results.


In [18]:
negative_discount = df["compare_price"] < df["current_price"]
zero_compare      = df["compare_price"] == 0
bad_rows          = negative_discount | zero_compare

print("Rows where compare_price < current_price (negative discount):", int(negative_discount.sum()))
print("Rows where compare_price == 0 (undefined discount):          ", int(zero_compare.sum()))
print("Total illogical rows to drop:                                ", int(bad_rows.sum()))

if bad_rows.sum() > 0:
    print("\nSamples of dropped rows:")
    print(df.loc[bad_rows, ["title", "current_price", "compare_price"]].head(10).to_string())

before = len(df)
df = df[~bad_rows].reset_index(drop=True)
print(f"\nDropped {before - len(df)} rows. Remaining: {len(df)}")

# Calculate discount (safe now — compare_price is either >= current_price or NaN)
df["discount_pct"] = (
    (df["compare_price"] - df["current_price"]) / df["compare_price"] * 100
).round(1)


Rows where compare_price < current_price (negative discount): 400
Rows where compare_price == 0 (undefined discount):           234
Total illogical rows to drop:                                 401

Samples of dropped rows:
                                                                     title  current_price  compare_price
403                                      How The Marquis Got His Coat Back          12.90          11.94
2188                                       Disney 12-Book Collection Boxes          89.90          79.90
4892                                             PUFFIN: NEVERENDING STORY          19.90          17.90
7899                                              Barron's First Thesaurus          17.90           6.45
17220                                                         Fast Cooking         199.90         110.00
17336                                                       Getting To Yes          74.90          46.90
19494  Not What I Expected: Help And Hope

## 8. Final checks and save

Hard assertions that **must** all pass before we write the output file.

In [19]:
# --- Hard assertions ---
assert (df["current_price"] >= 0).all(), \
    "Found negative current_price"
assert (df["compare_price"].dropna() > 0).all(), \
    "Found zero or negative compare_price"
assert (df["discount_pct"].dropna() >= 0).all(), \
    "Found negative discount_pct"
assert (df["discount_pct"].dropna() <= 100).all(), \
    "Found discount_pct > 100 (impossible)"
assert df["title"].str.contains(r"^\[Bargain", case=False, na=False).sum() == 0, \
    "Some titles still start with [Bargain Corner]"
assert df["author"].dropna().apply(is_valid_name).all(), \
    "Found author values with no letters"
assert df["publisher"].dropna().apply(is_valid_name).all(), \
    "Found publisher values with no letters"
assert (df["created_at"].notna()).all(), \
    "Some created_at values are missing"

print("All hard checks passed.")
print()
print("Final shape:", df.shape)
print("Final columns:", df.columns.tolist())
print()
df.dropna(inplace=True) # Final safety check: drop any remaining rows with missing values.
print("Missing values per column:")  
print(df.isna().sum())
print()
df.head(3)

All hard checks passed.

Final shape: (147582, 13)
Final columns: ['product_id', 'title', 'author', 'publisher', 'sku', 'current_price', 'compare_price', 'grams', 'tags', 'image_url', 'created_at', 'description', 'discount_pct']

Missing values per column:
product_id       0
title            0
author           0
publisher        0
sku              0
current_price    0
compare_price    0
grams            0
tags             0
image_url        0
created_at       0
description      0
discount_pct     0
dtype: int64



,product_id,title,author,publisher,sku,current_price,compare_price,grams,tags,image_url,created_at,description,discount_pct
0,6978035122382,Visions,Kelley Armstrong,Sphere,9780751547238,17.9,62.90,355,"[2022], BXOS, F CTM, Fiction, Kelley Armstrong...",https://cdn.shopify.com/s/files/1/0601/0883/29...,2022-04-01 10:46:08+08:00,"Olivia Jones is smart, capable, loyal...and th...",71.5
1,6978035318990,The Bone Bed,Patricia Cornwell,Sphere,9780751548174,17.9,43.95,360,"[2022], BXOS, F CTM, Fiction, Paperback, Patri...",https://cdn.shopify.com/s/files/1/0601/0883/29...,2022-04-01 10:46:08+08:00,A woman has vanished while digging a dinosaur ...,59.3
2,6978035417294,Jasmine Skies,Sita Brahmachari,MacMillan Children's Books,9781447205180,12.9,32.95,340,"[2022], BXOS, F YA, Fiction, Paperback, RM 10 ...",https://cdn.shopify.com/s/files/1/0601/0883/29...,2022-04-01 10:46:09+08:00,Mira Levenson is bursting with excitement as s...,60.8


In [20]:
FINAL_COLUMNS = [
    "product_id",
    "sku",
    "title",
    "author",
    "publisher",
    "current_price",
    "compare_price",
    "discount_pct",
    "grams",
    "tags",
    "image_url",
    "created_at",
    "description",
]

df_out = df[FINAL_COLUMNS]

df_out.to_csv(
    OUTPUT_FILE,
    index=False,
)

print(f"Wrote {len(df_out):,} rows to {OUTPUT_FILE}")

Wrote 147,558 rows to cleaned_data.csv


## Schema — `cleaned_data.csv`

| Column | Type | Notes |
|---|---|---|
| product_id | int | Shopify product ID |
| sku | string | ISBN-13 mostly; a few rows have no SKU |
| title | string | `[Bargain Corner]` prefix removed |
| author | string | Renamed from `Product Type`; dangling punctuation stripped; invalid (non-name) rows dropped |
| publisher | string | Renamed from `Vendor`; same cleaning as author |
| current_price | float | MYR |
| compare_price | float | MYR; NaN if no list price was set |
| discount_pct | float | `(compare - current) / compare × 100`; always ≥ 0; NaN where compare_price is absent |
| grams | int | Product weight |
| tags | string | Comma-separated tags; whitespace and entities cleaned |
| image_url | string | Shopify CDN URL |
| created_at | datetime | Asia/Kuala_Lumpur (UTC+08:00) |
| description | string | Plain text extracted from `Body HTML` |
